[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/MOOSE/Notebooks/moosePostProcessNotebook.ipynb)

# **MOOSE Post-Processing (Task 2): Convert NIfTI segmentations to DICOM-SEG**

This notebook is the CPU-side of the MOOSE twoVM workflow. It:
1. Extracts the `moose_segmentations.tar.lz4` archive produced by Task 1
2. Re-downloads source DICOM from IDC for each series (needed by dcmqi as reference)
3. For each series/model pair, builds a dcmqi config JSON and runs `itkimage2segimage`
4. Tars + lz4-compresses all DICOM-SEG files as `moose_dicom_seg.tar.lz4`

Please cite:

Herz C, Fillion-Robin JC, Onken M, Riesmeier J, Lasso A, Pinter C, Fichtinger G, Pieper S, Clunie D, Kikinis R, Fedorov A. dcmqi: An Open Source Library for Standardized Communication of Quantitative Image Analysis Results Using DICOM. Cancer Res. 2017 Nov 1;77(21):e87-e90. https://doi.org/10.1158/0008-5472.CAN-17-0336

Shiyam Sundar LK, et al. Fully automated, semantic segmentation of whole-body 18F-FDG PET/CT images based on data-centric artificial intelligence. J Nucl Med. 2022. https://doi.org/10.2967/jnumed.122.264063

## Imports

In [ ]:
import importlib.util
import json
import os
import re
import shutil
import subprocess
import time
import traceback
from pathlib import Path

import nibabel as nib
import numpy as np
from idc_index.index import IDCClient


## Parameters

Tagged `parameters` so papermill can override via `-p segmentationArchivePath <path>`.

In [ ]:
# Path to the lz4-compressed tar produced by Task 1 (mooseInferenceNotebook)
segmentationArchivePath = "moose_segmentations.tar.lz4"


## SNOMED mappings and filename utilities


In [ ]:
# Load SNOMED mappings from MOOSE's own source file (wget'd by the WDL).
# moose_to_snomed: label_name -> {"ID": snomed_code, "name": snomed_description}
_snomed_spec = importlib.util.spec_from_file_location("SNOMED", "SNOMED.py")
_snomed_mod = importlib.util.module_from_spec(_snomed_spec)
_snomed_spec.loader.exec_module(_snomed_mod)
moose_to_snomed = _snomed_mod.moose_to_snomed
print(f"Loaded {len(moose_to_snomed)} SNOMED mappings")

# Regex to extract moosez model identifier from segmentation filenames.
# e.g. "clin_CT_organs_segmentation_CT_foo.nii.gz" -> "clin_ct_organs"
_MOOSE_SEG_FILENAME_RE = re.compile(
    r"^((?:clin|preclin)_[A-Z0-9_-]+?_[a-z][a-z0-9_]*)_segmentation_"
)


def model_from_filename(filename: str) -> str:
    m = _MOOSE_SEG_FILENAME_RE.match(filename)
    return m.group(1).lower() if m else ""


## Extract the segmentation archive

In [ ]:
EXTRACT_DIR = Path("/tmp/moose_extract")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

subprocess.run(
    f"lz4 -d -c {segmentationArchivePath} | tar -xf - -C {EXTRACT_DIR}",
    shell=True,
    check=True,
)

top_dirs = [p for p in EXTRACT_DIR.iterdir() if p.is_dir()]
if len(top_dirs) == 1:
    MOOSE_ROOT = top_dirs[0]
else:
    MOOSE_ROOT = EXTRACT_DIR

# Load organ indices bundled by the inference step.
# organ_indices: {model_name: {label_int: label_name}}
_indices_path = MOOSE_ROOT / "moose_organ_indices.json"
if _indices_path.exists():
    _raw = json.loads(_indices_path.read_text())
    organ_indices = {model: {int(k): v for k, v in labels.items()}
                     for model, labels in _raw.items()}
    print(f"Loaded organ indices for {len(organ_indices)} model(s): {list(organ_indices)}")
else:
    organ_indices = {}
    print("WARNING: moose_organ_indices.json not found — unknown segments will use generic codes")

series_dirs = sorted([p for p in MOOSE_ROOT.iterdir() if p.is_dir()])
print(f"Found {len(series_dirs)} series in {MOOSE_ROOT}")
for p in series_dirs:
    print(f"  {p.name}")


## Helpers

In [ ]:
idc_client = IDCClient()

MODEL_DIR_RE = re.compile(r"^moosez-(?P<model>.+?)-\d{4}[-_]?\d{2}[-_]?\d{2}[-_T]?\d{2}[-_:]?\d{2}[-_:]?\d{2}$")

_ANATOMICAL_STRUCTURE = {
    "CodeValue": "123037004",
    "CodingSchemeDesignator": "SCT",
    "CodeMeaning": "Anatomical Structure",
}
_GENERIC_TYPE = {
    "CodeValue": "246183007",
    "CodingSchemeDesignator": "SCT",
    "CodeMeaning": "Body structure",
}


def organ_name(model: str, label_id: int) -> str:
    return organ_indices.get(model, {}).get(label_id, f"segment_{label_id}")


def download_dicom(uid: str, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    idc_client.download_from_selection(
        downloadDir=str(dest),
        seriesInstanceUID=uid,
    )


def find_series_number(dicom_dir: Path) -> str:
    try:
        import pydicom
        for p in dicom_dir.rglob("*.dcm"):
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
            sn = getattr(ds, "SeriesNumber", None)
            if sn is not None:
                return str(int(sn))
    except Exception:
        pass
    return "1"


def build_dcmqi_config(model: str, labels: list, series_number: str) -> dict:
    segments = []
    for label_id in labels:
        name = organ_name(model, int(label_id))
        entry = moose_to_snomed.get(name)
        if entry:
            display_name = entry["name"]
            seg_type = {
                "CodeValue": entry["ID"],
                "CodingSchemeDesignator": "SCT",
                "CodeMeaning": entry["name"],
            }
        else:
            display_name = name
            seg_type = _GENERIC_TYPE
        segments.append({
            "labelID": int(label_id),
            "SegmentDescription": display_name,
            "SegmentLabel": display_name,
            "SegmentAlgorithmType": "AUTOMATIC",
            "SegmentAlgorithmName": "moosez",
            "SegmentedPropertyCategoryCodeSequence": _ANATOMICAL_STRUCTURE,
            "SegmentedPropertyTypeCodeSequence": seg_type,
            "recommendedDisplayRGBValue": [128, 128, 128],
        })

    return {
        "ContentCreatorName": "MOOSE^CloudSegmentator",
        "ClinicalTrialSeriesID": "Session1",
        "ClinicalTrialTimePointID": "1",
        "SeriesDescription": f"MOOSE ({model}) Segmentation",
        "SeriesNumber": str(int(series_number) * 100 + 1) if series_number.isdigit() else "100",
        "InstanceNumber": "1",
        "BodyPartExamined": "",
        "segmentationType": "LABELMAP",
        "segmentAttributes": [segments],
        "ContentLabel": "SEGMENTATION",
        "ContentDescription": f"moosez {model} multi-label segmentation",
        "ClinicalTrialCoordinatingCenterName": "",
    }


def unique_nonzero_labels(nifti_path: Path) -> list:
    arr = np.asanyarray(nib.load(str(nifti_path)).dataobj)
    return sorted(int(v) for v in np.unique(arr) if v != 0)


## Generate DICOM-SEG per series/model

In [ ]:
DICOM_DIR = Path("/tmp/dicom")
DICOM_SEG_DIR = Path("/tmp/moose_dicom_seg")
CONFIG_DIR = Path("/tmp/moose_configs")
for d in (DICOM_DIR, DICOM_SEG_DIR, CONFIG_DIR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

dicom_seg_errors = []
usage_metrics = {"series": {}}

for series_dir in series_dirs:
    uid = series_dir.name
    print(f"\n=== Processing {uid} ===")
    series_start = time.time()

    dicom_dest = DICOM_DIR / uid
    try:
        t0 = time.time()
        download_dicom(uid, dicom_dest)
        usage_metrics["series"].setdefault(uid, {})["download_s"] = round(time.time() - t0, 1)
    except Exception as exc:
        msg = f"{uid}: download failed: {exc}\n{traceback.format_exc()}"
        dicom_seg_errors.append(msg)
        print(f"ERROR: {msg}")
        continue

    series_number = find_series_number(dicom_dest)

    seg_out_dir = DICOM_SEG_DIR / uid
    seg_out_dir.mkdir(parents=True, exist_ok=True)

    model_dirs = [p for p in series_dir.iterdir() if p.is_dir()]
    synthetic_model_dirs = False
    if not model_dirs:
        flat_seg_files = sorted(series_dir.rglob("*.nii.gz"))
        if flat_seg_files:
            synthetic_model_dirs = True
            model_dirs = [series_dir]
            print(f"INFO: {uid}: using legacy flat segmentation layout ({len(flat_seg_files)} files)")
        else:
            msg = f"{uid}: no model directories or .nii.gz segmentations found in {series_dir}"
            dicom_seg_errors.append(msg)
            print(f"WARN: {msg}")
            continue

    for model_dir in model_dirs:
        if synthetic_model_dirs:
            model = "legacy"
            seg_files = sorted(model_dir.rglob("*.nii.gz"))
        else:
            m = MODEL_DIR_RE.match(model_dir.name)
            model = m.group("model") if m else model_dir.name
            seg_files = list((model_dir / "segmentations").glob("*.nii.gz")) \
                if (model_dir / "segmentations").exists() \
                else list(model_dir.rglob("*.nii.gz"))

        if not seg_files:
            msg = f"{uid}/{model}: no .nii.gz segmentations under {model_dir}"
            dicom_seg_errors.append(msg)
            print(f"WARN: {msg}")
            continue

        for seg_idx, seg_file in enumerate(seg_files):
            try:
                labels = unique_nonzero_labels(seg_file)
                if not labels:
                    print(f"  {model}/{seg_file.name}: empty mask, skipping")
                    continue

                effective_model = model
                if effective_model not in organ_indices:
                    fn_model = model_from_filename(seg_file.name)
                    if fn_model:
                        effective_model = fn_model

                cfg = build_dcmqi_config(effective_model, labels, series_number)
                cfg_path = CONFIG_DIR / f"{uid}_{effective_model}_{seg_idx}.json"
                cfg_path.write_text(json.dumps(cfg, indent=2))

                out_dcm = seg_out_dir / f"{effective_model}_{seg_idx}.dcm"
                t0 = time.time()
                result = subprocess.run(
                    [
                        "itkimage2segimage",
                        "--inputImageList", str(seg_file),
                        "--inputDICOMDirectory", str(dicom_dest),
                        "--outputDICOM", str(out_dcm),
                        "--inputMetadata", str(cfg_path),
                        "--segmentationType", "labelmap",
                        "--skip",
                    ],
                    capture_output=True,
                    text=True,
                )
                elapsed = round(time.time() - t0, 1)

                if result.returncode != 0 or not out_dcm.exists():
                    msg = (
                        f"{uid}/{model}/{seg_file.name}: itkimage2segimage failed "
                        f"(rc={result.returncode})\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
                    )
                    dicom_seg_errors.append(msg)
                    print(f"  ERROR: {msg}")
                else:
                    usage_metrics["series"].setdefault(uid, {}).setdefault("models", {})[effective_model] = elapsed
                    print(f"  {effective_model}: wrote {out_dcm.name} ({elapsed}s, {len(labels)} segments)")

            except Exception as exc:
                msg = f"{uid}/{model}/{seg_file.name}: {traceback.format_exc()}"
                dicom_seg_errors.append(msg)
                print(f"  ERROR: {exc}")

    usage_metrics["series"].setdefault(uid, {})["total_s"] = round(time.time() - series_start, 1)
    shutil.rmtree(dicom_dest, ignore_errors=True)

if dicom_seg_errors:
    Path("dicom_seg_error_file.txt").write_text("\n\n".join(dicom_seg_errors))


## Package DICOM-SEG outputs

In [ ]:
def compress_dir(src_dir: Path, out_file: str) -> None:
    cmd = f"tar -cf - -C {src_dir.parent} {src_dir.name} | lz4 > {out_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Compression failed: {result.stderr}")
    size_mb = Path(out_file).stat().st_size / (1024 ** 2)
    print(f"Created {out_file} ({size_mb:.1f} MB)")

compress_dir(DICOM_SEG_DIR, "moose_dicom_seg.tar.lz4")

## Write usage metrics

In [ ]:
metrics_json = json.dumps(usage_metrics, indent=2)
metrics_path = Path("moose_postprocess_UsageMetrics.json")
metrics_path.write_text(metrics_json)

subprocess.run(
    f"lz4 -f {metrics_path} moose_postprocess_UsageMetrics.lz4",
    shell=True,
    check=True,
)

print(metrics_json)

## Summary

In [ ]:
print("=" * 60)
print("MOOSE Post-Process Summary")
print("=" * 60)
print(f"  Series processed : {len(series_dirs)}")
print(f"  Errors logged    : {len(dicom_seg_errors)}")
print("=" * 60)

if len(dicom_seg_errors) and len(dicom_seg_errors) >= len(series_dirs):
    raise RuntimeError("DICOM-SEG generation failed for all series - see dicom_seg_error_file.txt")